# Part B – Question 2: Logistic Regression from Scratch on FashionMNIST

We implement **softmax (multinomial logistic) regression** entirely from scratch using only NumPy — no PyTorch, no sklearn for the learning algorithm.

Steps:
1. Load FashionMNIST via `datasets`
2. Implement forward pass (softmax), cross-entropy loss, and gradient computation manually
3. Train with vanilla (batch) gradient descent
4. Evaluate accuracy on the test set

In [ ]:
import numpy as np
from datasets import load_dataset
import matplotlib.pyplot as plt

## 1. Load FashionMNIST

In [ ]:
print("Loading FashionMNIST...")
fmnist = load_dataset('fashion_mnist')

def extract(split, n=None):
    """Return (X, y) as float32 numpy arrays. X is normalised to [0,1]."""
    data = fmnist[split]
    if n is not None:
        data = data.select(range(n))
    # Each 'image' is a PIL Image; convert via numpy
    images = np.array([np.array(img).flatten() for img in data['image']], dtype=np.float32)
    images /= 255.0
    labels = np.array(data['label'], dtype=np.int32)
    return images, labels

# Use 10 000 training samples to keep runtime reasonable for a naive implementation
X_train, y_train = extract('train', n=10000)
X_test,  y_test  = extract('test')

print(f"Train: {X_train.shape},  Test: {X_test.shape}")
print(f"Labels: {np.unique(y_train)}")

## 2. Logistic Regression from Scratch

### Model

$$\hat{y} = \text{softmax}(XW + b), \quad W \in \mathbb{R}^{784 \times 10}, \; b \in \mathbb{R}^{10}$$

### Loss (cross-entropy)

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \log \hat{p}_{i, y_i}$$

### Gradients

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{1}{n} X^T (\hat{P} - Y_{\text{onehot}}), \quad \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{n} \sum_i (\hat{p}_i - y_{i,\text{onehot}})$$

In [ ]:
def softmax(z):
    """Numerically stable softmax. z shape: (n, C)."""
    z_shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def one_hot(y, n_classes=10):
    """Return one-hot matrix of shape (n, n_classes)."""
    n = len(y)
    oh = np.zeros((n, n_classes), dtype=np.float32)
    oh[np.arange(n), y] = 1.0
    return oh


def cross_entropy_loss(probs, y):
    """Mean cross-entropy loss. probs: (n, C), y: (n,) integer labels."""
    n = len(y)
    # Clip to avoid log(0)
    correct_probs = probs[np.arange(n), y].clip(1e-12, 1.0)
    return -np.mean(np.log(correct_probs))


def compute_gradients(X, probs, y):
    """Return grad_W, grad_b."""
    n = len(y)
    delta = probs - one_hot(y)   # (n, C)
    grad_W = X.T @ delta / n     # (784, C)
    grad_b = delta.mean(axis=0)  # (C,)
    return grad_W, grad_b


def accuracy(probs, y):
    return np.mean(probs.argmax(axis=1) == y)


print("Helper functions defined.")

## 3. Gradient Descent Training

In [ ]:
np.random.seed(0)
n_classes = 10
n_features = X_train.shape[1]   # 784

# Initialise weights small
W = np.random.randn(n_features, n_classes).astype(np.float32) * 0.01
b = np.zeros(n_classes, dtype=np.float32)

# Hyperparameters
lr      = 0.5     # learning rate
n_iters = 200     # gradient descent iterations

train_losses = []
test_accs    = []

for i in range(n_iters):
    # Forward pass
    logits = X_train @ W + b         # (n, 10)
    probs  = softmax(logits)

    # Loss
    loss = cross_entropy_loss(probs, y_train)
    train_losses.append(loss)

    # Gradients
    grad_W, grad_b = compute_gradients(X_train, probs, y_train)

    # Update
    W -= lr * grad_W
    b -= lr * grad_b

    # Test accuracy every 20 steps
    if (i + 1) % 20 == 0:
        test_logits = X_test @ W + b
        test_probs  = softmax(test_logits)
        acc = accuracy(test_probs, y_test)
        test_accs.append(acc)
        print(f"Iter {i+1:4d} | Loss: {loss:.4f} | Test Acc: {acc*100:.2f}%")

print("\nTraining complete.")

## 4. Final Evaluation on Test Set

In [ ]:
test_logits = X_test @ W + b
test_probs  = softmax(test_logits)
final_acc   = accuracy(test_probs, y_test)

print(f"Final Test Accuracy: {final_acc * 100:.2f}%")

# Class names for FashionMNIST
class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

# Per-class accuracy
preds = test_probs.argmax(axis=1)
print("\nPer-class accuracy:")
for c in range(n_classes):
    mask = y_test == c
    cls_acc = np.mean(preds[mask] == c)
    print(f"  {class_names[c]:15s}: {cls_acc*100:.1f}%")

## 5. Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')

axes[1].plot(range(20, n_iters + 1, 20), [a * 100 for a in test_accs], marker='o')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy over Training')

plt.tight_layout()
plt.savefig('q2_training_curve.png', dpi=100)
plt.show()